In [ ]:
# imports
import numpy as np
from iqpy.iqp_to_qiskit import IqpCircuitQiskit
from iqpy.utils import median_heuristic_fast, generate_experiment_mixture_dataset, generate_simple_dataset
from iqpy.ansatzes import fully_connected_IQP_ansatz, nearest_neighbour_from_graph_IQP_ansatz, nearest_neighbour_IQP_ansatz
from iqpy.torch_training import TrainerTorch
from iqpy.torch_methods import mmd_loss_torch
from qiskit import qpy
import networkx as nx

In [ ]:
n_qubits = 5  # number of qubits in the circuit (max 53 for the Q50 machine)

In [ ]:
# constructs the graph of the Q50 machine if nearest neighbour gates are used

edges = [
    (0, 1), (0, 4), (3, 2), (3, 4), (3, 9), (5, 1), (5, 4), (5, 6), (5, 11), (8, 2), (8, 7), (8, 9), (8, 16), (10, 4), (10, 9), (10, 11), (10, 18), (12, 6), (12, 11), (12, 13), (12, 20), (15, 7), (15, 14), (15, 16), (15, 23), (17, 16), (17, 18), (17, 25), (19, 20), (21, 13), (21, 20), (21, 29), (22, 14), (22, 23), (24, 16), (24, 23), (24, 25), (24, 31), (26, 18), (26, 25), (26, 27), (26, 33), (28, 20), (28, 27), (28, 29), (28, 35), (30, 29), (30, 37), (32, 25), (32, 31), (32, 33), (32, 40), (34, 27), (34, 33), (34, 35), (34, 42), (36, 29), (36, 35), (36, 37), (36, 44), (39, 31), (39, 38), (39, 40), (39, 45), (41, 33), (41, 40), (41, 42), (41, 47), (43, 35), (43, 44), (43, 49), (46, 40), (46, 45), (46, 47), (46, 50), (48, 42), (48, 47), (48, 49), (48, 52), (51, 47), (51, 50), (51, 52),
]

G = nx.Graph()
G.add_nodes_from(range(n_qubits))

G.add_edges_from(
    edge for edge in edges if edge[0] < n_qubits - 1 and edge[1] < n_qubits - 1
)

In [ ]:
# run this to use a fully connected ansatz
gates = fully_connected_IQP_ansatz(n_qubits)

# run this to use a nearest neighbour ansatz (not based on the topology)
# gates = nearest_neighbour_IQP_ansatz(n_qubits)

# run this to use a nearest neighbour ansatz based on the topology of the Q50 machine
# max_distance = 2 # the maximum distance between nodes to consider when creating gates
# gates = nearest_neighbour_from_graph_IQP_ansatz(G, max_distance)

In [ ]:
n_ops = 200
n_samples = 2500  # number of samples to train on

In [ ]:
# run this to use a mixture dataset
X_train = generate_experiment_mixture_dataset(n_qubits, n_samples=n_samples)

# run this to use a simple dataset with few 1s
# X_train = generate_simple_dataset(n_qubits, n_samples=n_samples)

print(f"Training data: {len(X_train)} samples")

In [ ]:
# training

circuit = IqpCircuitQiskit(n_qubits, gates)

sigma = median_heuristic_fast(X_train, n_samples=n_samples)

params_init = np.random.normal(0, 1 / np.sqrt(n_qubits), len(gates))
p = (1 - np.exp(-1 / (2 * sigma**2))) / 2
ops = np.random.binomial(1, p, size=(n_ops, n_qubits))
n_iqp_samples = n_samples

loss_kwargs = {
    "params": params_init,
    "circuit": circuit,
    "ground_truth": X_train,
    "ops": ops,
    "n_samples": n_iqp_samples,
}

trainer = TrainerTorch(mmd_loss_torch, lr=0.01)
trainer.train(n_iters=100, loss_kwargs=loss_kwargs)

final_circuit = circuit.iqp_circuit(trainer.final_params)
final_circuit.measure_all()
with open("circuit.qpy", "wb") as file:
    qpy.dump([final_circuit], file)
print("Wrote circuit to circuit.qpy")